# Day 2 — 보고서 전략의 구현과 모델 평가
Day 1 EDA의 모델 설계 전략에 맞춰 선택 기준을 반영한 실험입니다.  
**전략 → 구현, Pipeline·분할, 성능·Gap, 도메인 해석·한계**를 순서대로 확인합니다.  
Batch 1 정책 그룹 Hold-out을 먼저 분리하고, 남은 셀의 그룹 CV로 후보를 선택합니다. Hold-out 평가 후 설정을 고정해 Batch 1 전체로 재학습하고 Batch 2를 평가합니다.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

## 1. 모델 후보와 선택 규칙

중앙값 기준선, 선형 회귀, Ridge·ElasticNet, 얕은 Random Forest·Gradient Boosting을 원래/로그 수명과 비교합니다. 로그 예측은 원래 사이클 단위로 되돌려 MAPE를 계산합니다.

Day1의 "개선이 작거나 폴드마다 방향이 다르면 단순한 묶음 선택"을 다음과 같이 수치화했습니다. **추가 피처는 같은 모델·타깃의 기준 묶음보다 평균 MAPE가 1%p 이상 낮고 모든 CV 폴드에서 개선될 때 채택**합니다.  
추가 용량 변화는 B와, 정책 결합항은 D와 비교하며 부모 묶음도 채택 기준을 통과해야 합니다.  
ΔQ 대표 피처 교체는 피처 수가 같은 후보로 비교하며, 통과한 후보 중 평균 CV MAPE가 가장 낮은 모델을 선택합니다.

In [2]:
from src.train import run_modeling, performance_report, MIN_IMPROVEMENT_PP
features = pd.read_csv(ROOT / 'results/features.csv')
performance, model, selection = run_modeling(features, include_batch3=False)
report = performance_report(performance)
display(report.round(2))

CV feature bundle: A
CV feature bundle: B
CV feature bundle: C
CV feature bundle: D
CV feature bundle: E
CV feature bundle: B_QD_change
CV feature bundle: B_QD_relative
CV feature bundle: C_Tmax
CV feature bundle: D_interaction
CV feature bundle: A_delta_min
CV feature bundle: A_delta_mean


,구분,MAPE (%),비고
0,Train (Batch 1 CV),8.40,모델 선택에 사용한 그룹 CV 평균; 독립 성능 추정 아님
1,Valid (Batch 1 Hold-out),6.52,CV와 정책 그룹 중복 없이 분리; 재조정에 사용하지 않음
2,Test (Batch 2),26.09,설정 고정 후 Batch 1 전체 재학습
3,Gap (Train-Valid),-1.88,Valid−Train (%p); (+): 과적합 의심
4,Gap (Valid-Test),19.57,Test−Valid (%p); (+): 배치 간 일반화 저하 의심
5,Gap (Target-Test),16.99,Test−9.1 (%p); Target: 원논문 비교 목표 9.1%


## 2. 셀·정책 그룹 분할 검증

같은 수치 정책 (C1, 전환 비율, C2)의 셀은 같은 그룹으로 묶으며, 미해석 정책은 원문을 그룹 키로 사용합니다.  
결측 대체·표준화는 각 학습 폴드 안에서 적합합니다. 피처 묶음 선택은 Hold-out을 제외한 Batch 1 그룹 CV 안에서 수행하며 별도의 중첩 CV는 아닙니다.

In [3]:
split = pd.read_csv(ROOT / 'results/batch1_split.csv')
assignments = pd.read_csv(ROOT / 'results/cv_assignments.csv')
assert split.cell_id.is_unique
assert set(split.loc[split.split.eq('CV'), 'policy_group']).isdisjoint(
    split.loc[split.split.eq('Hold-out'), 'policy_group'])
for _, fold in assignments.groupby('fold'):
    assert set(fold.loc[fold.role.eq('train'), 'policy_group']).isdisjoint(
        fold.loc[fold.role.eq('valid'), 'policy_group'])
display(split.groupby('split').agg(cells=('cell_id', 'size'),
                                   policy_groups=('policy_group', 'nunique')))
display(assignments.groupby(['fold', 'role']).agg(cells=('cell_id', 'size'),
                                                 policy_groups=('policy_group', 'nunique')))

,cells,policy_groups
split,,
CV,36,18
Hold-out,10,5


cells  policy_groups
fold role                       
1    train     29             15
     valid      7              3
2    train     28             14
     valid      8              4
3    train     28             14
     valid      8              4
4    train     29             14
     valid      7              4
5    train     30             15
     valid      6              3

## 3. 후보 채택·탈락 근거와 최종 모델

단순한 평균 순위와 실제 채택 기준을 함께 확인합니다. `eligible`은 평균 개선과 전 폴드 개선을 만족하는지, `selected`는 최종 선택인지 나타냅니다.  
`improvement_pp`는 기준 후보 MAPE−현재 후보 MAPE로 양수이면 개선입니다.

In [4]:
selected = pd.read_csv(ROOT / 'results/selected_model.csv').iloc[0]
display(selection.sort_values('CV_MAPE')[['bundle', 'model', 'log_target', 'CV_MAPE',
    'CV_std', 'reference', 'improvement_pp', 'improved_folds', 'eligible', 'selected']].head(20).round(3))
display(pd.DataFrame([selected]))
display(pd.read_csv(ROOT / 'results/selected_cv_folds.csv').round(3))
display(Markdown(f"선택 모델은 **{selected.model}**, 묶음 **{selected.bundle}**, "
    f"타깃은 **{'로그 수명' if selected.log_target else '원래 수명'}**입니다. "
    f"입력: `{selected.features}`. 설정: `{selected.parameters}`."))
# 채택 가능한 후보와 평균 점수만 가장 낮았던 후보를 구분합니다.
raw_best = selection.loc[selection.CV_MAPE.idxmin()]
if not raw_best.eligible:
    display(Markdown(f"평균 CV MAPE만 가장 낮았던 후보는 **{raw_best.bundle} / {raw_best.model}** "
        f"({raw_best.CV_MAPE:.2f}%)입니다. 기준 묶음보다 평균 {raw_best.improvement_pp:.2f}%p "
        f"개선됐지만 개선 폴드가 {raw_best.improved_folds}/5여서 채택하지 않았습니다."))
display(Markdown("최종 모델은 채택 기준을 통과한 후보 중 CV MAPE가 가장 낮아 선택했습니다. "
    "ΔQ 최솟값·평균은 보고서에서 계획한 대표 통계 교체 후보이며 로그 분산과 중복 입력하지 않습니다. "
    "Ridge의 규제는 작은 표본에서 계수를 제어하는 수단이고 배치 간 일반화를 보장하지 않습니다."))


,bundle,model,log_target,CV_MAPE,CV_std,reference,improvement_pp,improved_folds,eligible,selected
21,B,GradientBoosting,True,7.671,1.625,A,1.415,3,False,False
20,B,GradientBoosting,False,7.859,1.714,A,1.207,4,False,False
41,D,GradientBoosting,True,8.143,0.985,A,0.943,4,False,False
51,E,GradientBoosting,True,8.327,2.081,A,0.759,3,False,False
61,B_QD_change,GradientBoosting,True,8.353,1.684,B,-0.682,0,False,False
71,B_QD_relative,GradientBoosting,True,8.356,1.556,B,-0.685,0,False,False
30,C,GradientBoosting,False,8.383,1.613,A,0.684,1,False,False
95,A_delta_min,Ridge,True,8.404,1.026,,NaN,0,True,True
97,A_delta_min,ElasticNet,True,8.405,1.027,,NaN,0,True,False
93,A_delta_min,Linear,True,8.409,1.033,,NaN,0,True,False


,bundle,model,log_target,features,parameters,selection_reason
0,A_delta_min,Ridge,True,delta_min,{'regressor__model__alpha': 0.1},기본 또는 대표 피처 교체 후보


,fold,MAPE,MAE,RMSE
0,1,6.662,70.126,93.151
1,2,7.786,68.134,85.289
2,3,9.193,74.892,87.439
3,4,9.184,74.018,95.649
4,5,9.192,71.959,97.982


선택 모델은 **Ridge**, 묶음 **A_delta_min**, 타깃은 **로그 수명**입니다. 입력: `delta_min`. 설정: `{'regressor__model__alpha': 0.1}`.

평균 CV MAPE만 가장 낮았던 후보는 **B / GradientBoosting** (7.67%)입니다. 기준 묶음보다 평균 1.41%p 개선됐지만 개선 폴드가 3/5여서 채택하지 않았습니다.

최종 모델은 채택 기준을 통과한 후보 중 CV MAPE가 가장 낮아 선택했습니다. ΔQ 최솟값·평균은 보고서에서 계획한 대표 통계 교체 후보이며 로그 분산과 중복 입력하지 않습니다. Ridge의 규제는 작은 표본에서 계수를 제어하는 수단이고 배치 간 일반화를 보장하지 않습니다.

## 4. 지정 성능표와 목표 대비 Gap

주 지표는 MAPE(%), 보조 지표는 MAE·RMSE(사이클)입니다. Gap은 퍼센트포인트(%p)로, Valid−Train·Test−Valid·Test−9.1로 계산합니다. 작은 Hold-out에서 낮은 오차가 나와도 과적합이 없다고 확정할 수 없습니다.

In [5]:
display(report.round(2))
display(performance[~performance.Index.str.startswith('Gap')].rename(
    columns={'Index': '구분', 'MAPE': 'MAPE (%)', 'MAE': 'MAE (cycles)',
             'RMSE': 'RMSE (cycles)'}).round(2))
p = performance.set_index('Index')
test_mape = p.loc['Test (Batch 2)', 'MAPE']
gap = p.loc['Gap (Target-Test)', 'MAPE']
display(Markdown(f"Batch 2 MAPE는 **{test_mape:.2f}%**, 목표 대비 Gap은 "
    f"**{gap:+.2f}%p**입니다. 목표 9.1%를 {'충족했습니다' if gap <= 0 else '충족하지 못했습니다'}. "
    "9.1%는 과제의 비교 목표이며 원논문과 셀 선정·전처리·분할 조건이 동일한 재현 결과는 아닙니다."))

,구분,MAPE (%),비고
0,Train (Batch 1 CV),8.40,모델 선택에 사용한 그룹 CV 평균; 독립 성능 추정 아님
1,Valid (Batch 1 Hold-out),6.52,CV와 정책 그룹 중복 없이 분리; 재조정에 사용하지 않음
2,Test (Batch 2),26.09,설정 고정 후 Batch 1 전체 재학습
3,Gap (Train-Valid),-1.88,Valid−Train (%p); (+): 과적합 의심
4,Gap (Valid-Test),19.57,Test−Valid (%p); (+): 배치 간 일반화 저하 의심
5,Gap (Target-Test),16.99,Test−9.1 (%p); Target: 원논문 비교 목표 9.1%


,구분,MAPE (%),MAE (cycles),RMSE (cycles)
0,Train (Batch 1 CV),8.40,71.83,91.90
1,Valid (Batch 1 Hold-out),6.52,54.33,65.48
2,Test (Batch 2),26.09,132.26,154.95


Batch 2 MAPE는 **26.09%**, 목표 대비 Gap은 **+16.99%p**입니다. 목표 9.1%를 충족하지 못했습니다. 9.1%는 과제의 비교 목표이며 원논문과 셀 선정·전처리·분할 조건이 동일한 재현 결과는 아닙니다.

## 5. 큰 오차 셀과 수명 구간별 오류

절대 오차뿐 아니라 과대·과소 방향을 봅니다. MAPE는 같은 사이클 오차라도 수명이 짧을수록 커집니다. 오류 분석에는 전체 수명 라벨과 정책을 쓰지만 모델 입력에는 포함하지 않습니다.

In [6]:
predictions = pd.read_csv(ROOT / 'results/predictions.csv')
test = predictions[predictions.batch.eq('Batch 2')].copy()
test['absolute_error'] = abs(test.prediction - test.cycle_life)
test['APE'] = 100 * test.absolute_error / test.cycle_life
test['bias'] = test.prediction - test.cycle_life
test['life_group'] = np.select([test.cycle_life.lt(500), test.cycle_life.gt(1000)],
                               ['short <500', 'long >1000'], default='middle')
error = test.merge(features, on=['cell_id', 'batch', 'cycle_life'], validate='one_to_one')
by_life = error.groupby('life_group').agg(n=('cell_id', 'size'), MAPE=('APE', 'mean'),
    MAE=('absolute_error', 'mean'), mean_bias=('bias', 'mean'))
display(error.nlargest(10, 'APE')[['cell_id', 'policy', 'cycle_life', 'prediction',
                                'APE', 'bias', 'delta_log_var', 'QD_slope']].round(3))
display(by_life.round(2))
error.to_csv(ROOT / 'results/error_analysis.csv', index=False)
by_life.to_csv(ROOT / 'results/errors_by_life.csv')

,cell_id,policy,cycle_life,prediction,APE,bias,delta_log_var,QD_slope
18,B2_018,5.2C(50%)-4.25C,449.0,779.947,73.708,330.947,-3.707,0.0
6,B2_006,3.6C(9%)-5C,393.0,670.015,70.487,277.015,-3.530,0.0
15,B2_015,3.6C(9%)-5C,396.0,659.422,66.521,263.422,-3.516,0.0
2,B2_002,5.2C(50%)-4.25C,424.0,642.016,51.419,218.016,-3.460,-0.0
27,B2_029,5.2C(58%)-4C,452.0,642.828,42.219,190.828,-3.508,0.0
26,B2_028,4.8C(80%)-4.8C,437.0,606.679,38.828,169.679,-3.423,0.0
22,B2_024,4.8C(80%)-4.8C,514.0,712.976,38.711,198.976,-3.641,0.0
11,B2_011,5.2C(50%)-4.25C,449.0,621.349,38.385,172.349,-3.485,0.0
17,B2_017,5.6C(26%)-4.5C,471.0,643.515,36.627,172.515,-3.500,0.0
16,B2_016,4.8C(80%)-4.8C,472.0,636.869,34.930,164.869,-3.495,-0.0


,n,MAPE,MAE,mean_bias
life_group,,,,
long >1000,3,9.03,103.38,-81.32
middle,8,22.34,156.45,144.44
short <500,28,28.99,128.44,121.75


## 6. 배치 간 일반화 차이와 피처 이동

Batch 1 학습 수명 범위와 Batch 2의 단수명 구간을 비교합니다. 선택 입력의 학습 범위 안·밖과 `newstructure` 집단별 오류를 함께 확인합니다. 범위 밖은 분포 이동의 기술 통계이며 오류 원인을 입증하거나 셀을 제거하는 기준은 아닙니다. 범위 안에서도 피처–수명 관계가 달라질 수 있습니다.

In [7]:
batch1 = features[features.batch.eq('Batch 1') & features.cycle_life.notna()]
columns = selected.features.split(', ')
shift_rows = []
for column in columns:
    lower, upper = batch1[column].min(), batch1[column].max()
    outside = error[column].lt(lower) | error[column].gt(upper)
    for flag, label in [(False, '범위 안'), (True, '범위 밖')]:
        group = error[outside.eq(flag)]
        shift_rows.append(dict(feature=column, group=label, n=len(group),
            Batch1_min=lower, Batch1_max=upper, MAPE=group.APE.mean(),
            MAE=group.absolute_error.mean(), mean_bias=group.bias.mean()))
shift = pd.DataFrame(shift_rows)
display(shift.round(3))
display(error.groupby('newstructure').agg(n=('cell_id', 'size'),
    MAPE=('APE', 'mean'), mean_bias=('bias', 'mean')).round(2))
shift.to_csv(ROOT / 'results/errors_by_feature_range.csv', index=False)
short = error[error.cycle_life.lt(500)]
display(Markdown(f"Batch 1 수명 범위는 **{batch1.cycle_life.min():.0f}~{batch1.cycle_life.max():.0f}회**입니다. "
    f"Batch 2 단수명은 **{len(short)}셀**, MAPE **{short.APE.mean():.2f}%**, "
    f"평균 예측 편향 **{short.bias.mean():+.2f}회**입니다. "
    f"Batch 2 예측 범위는 **{error.prediction.min():.1f}~{error.prediction.max():.1f}회**입니다."))

,feature,group,n,Batch1_min,Batch1_max,MAPE,MAE,mean_bias
0,delta_min,범위 안,30,-0.065,-0.008,30.924,159.295,137.623
1,delta_min,범위 밖,9,-0.065,-0.008,9.985,42.130,21.309


,n,MAPE,mean_bias
newstructure,,,
0,30,29.26,124.85
1,9,15.53,63.89


Batch 1 수명 범위는 **534~1227회**입니다. Batch 2 단수명은 **28셀**, MAPE **28.99%**, 평균 예측 편향 **+121.75회**입니다. Batch 2 예측 범위는 **388.8~1062.4회**입니다.

### 원인 해석과 개선 방향

- **관측:** 내부 CV·Hold-out과 배치 테스트의 오차 차이, 수명 구간별 편향, 입력 범위·정책 집단별 오류를 위 표에서 확인합니다.
- **가설:** 학습에 없는 단수명 구간과 배치별 초기 상태·측정 조건·피처–수명 관계 변화가 일반화 저하에 기여했을 수 있습니다. 트리 모델은 범위 밖 예측에 제약이 있고 선형 외삽도 정확성을 보장하지 않습니다.
- **메타데이터 한계:** `newstructure`는 Batch 1에서 모두 0이므로 표기를 입력에 넣어도 표기 효과를 학습할 수 없습니다. 표기의 물리적 의미는 확인되지 않았습니다.
- **개선:** 실제 EOL 라벨과 측정 조건을 확인하고, 더 다양한 수명·운전 조건의 학습 셀을 확보한 뒤 새로운 독립 배치에서 검증합니다. 현재 Batch 2 점수가 낮은 후보로 모델을 교체하지 않습니다.

## 7. Day1에서 제안한 처리 가정·라벨 민감도

주 후보 비교에서 용량 변화·상대 변화·온도 교체·정책 결합항·ΔQ 대표 통계를 이미 비교했습니다. 아래는 선택 모델의 **알고리즘·설정·타깃 변환을 고정한 실험**입니다.  
A/B 용량, IR 원값/0 제외 가정, 충전시간, 강건/OLS 기울기, 높은 종료 용량 셀 제외를 확인합니다.

최종 후보 선정 이후 수행한 진단이며, 공식 모델·성능을 바꾸지는 않았습니다.

In [8]:
from src.train import run_auxiliary_experiments
aux_comparison, aux_folds, aux_paired = run_auxiliary_experiments(features)
display(aux_comparison.round(3))
display(aux_paired.round(3))
for row in aux_paired.itertuples():
    display(Markdown(f"- **{row.variant} vs {row.reference}:** CV 평균 차이 "
        f"{row.mean_gap_pp:+.2f}%p, 개선 폴드 {row.improved_folds}/{row.total_folds}."))

,variant,CV_MAPE,CV_std,Holdout_MAPE,n_holdout,n_refit,Batch2_MAPE,Batch2_MAE
0,A,11.938,4.006,7.228,10,46,34.890,170.224
1,B,10.098,5.166,8.016,10,46,56.821,278.635
2,B_QD_change,10.625,4.882,7.742,10,46,63.196,300.043
3,B_QD_relative,10.636,4.858,7.721,10,46,63.617,301.980
4,A_IR_raw,11.830,3.746,8.372,10,46,35.627,183.077
5,A_IR_nonzero,12.028,3.854,8.418,10,46,35.361,172.536
6,A_chargetime,12.385,3.934,7.315,10,46,36.942,179.969
7,B_OLS,9.395,6.103,9.548,10,46,49.474,246.214
8,endpoint_reference,9.298,1.584,8.290,7,46,26.092,132.257
9,endpoint_excluded,9.270,1.704,8.235,7,36,26.415,133.278


,variant,reference,improved_folds,total_folds,mean_gap_pp
0,B,A,4,5,-1.840
1,B_QD_change,B,2,5,0.527
2,B_QD_relative,B,2,5,0.538
3,A_IR_raw,A,3,5,-0.108
4,A_IR_nonzero,A_IR_raw,1,5,0.198
5,A_chargetime,A,1,5,0.447
6,B_OLS,B,4,5,-0.702
7,endpoint_excluded,endpoint_reference,3,5,-0.028


- **B vs A:** CV 평균 차이 -1.84%p, 개선 폴드 4/5.

- **B_QD_change vs B:** CV 평균 차이 +0.53%p, 개선 폴드 2/5.

- **B_QD_relative vs B:** CV 평균 차이 +0.54%p, 개선 폴드 2/5.

- **A_IR_raw vs A:** CV 평균 차이 -0.11%p, 개선 폴드 3/5.

- **A_IR_nonzero vs A_IR_raw:** CV 평균 차이 +0.20%p, 개선 폴드 1/5.

- **A_chargetime vs A:** CV 평균 차이 +0.45%p, 개선 폴드 1/5.

- **B_OLS vs B:** CV 평균 차이 -0.70%p, 개선 폴드 4/5.

- **endpoint_excluded vs endpoint_reference:** CV 평균 차이 -0.03%p, 개선 폴드 3/5.

보조 실험에서 음의 `mean_gap_pp`는 오차 감소를 뜻합니다. 평균과 폴드별 방향을 함께 해석하며, 고정된 설정에서 이득이 없었다고 모든 모델에서 무효라고 단정하지는 않습니다.

## 8. ESS 운영 관점의 활용과 한계

초기 셀 선별과 교체 계획의 참고 자료로 총수명 예측을 사용할 수 있습니다. 예측 총수명에서 100을 빼면 잔여 사이클 추정값을 만들 수 있지만, 이는 운전 조건과 실제 교체 날짜를 바로 의미하지 않습니다.

수명을 과대예측하면 교체를 늦춰 요구 용량·출력을 확보하지 못할 수 있고, 과소예측하면 사용 가능한 셀을 일찍 교체해 비용이 늘어날 수 있습니다. 따라서 위의 수명 구간별 편향과 절대 오차를 함께 봐야 합니다.

실제 ESS에는 온도·SOC·충전 조건 변화, 달력 열화, 셀 간 불균형, 팩 수준의 문제가 추가되는데, 현재 실험 셀의 결과만으로 자동 교체 결정을 지원할 근거는 부족합니다.  
따라서 라벨 의미와 배치별 측정 조건을 확인하고 실제 운전 데이터에서 예측 불확실성과 과대예측 위험을 검증해야 합니다. BMS의 SOC·SOH·온도와 함께 판단하며, 이 모델은 안전 이상이나 열폭주를 예측하지는 않습니다.

## 9. 재현 가능한 결과 파일

- `performance_report.csv`: 과제 지정 구분 / MAPE (%) / 비고 표.
- `model_performance.csv`: MAPE와 MAE·RMSE 원값, Gap.
- `model_selection.csv`, `selected_cv_folds.csv`: 후보별 선정 근거와 선택 모델의 폴드 오차.
- `batch1_split.csv`, `cv_assignments.csv`: 셀·정책 그룹 분할 기록.
- `selected_model.csv`, `run_metadata.json`: 선택 설정과 고정 기준, 실험 한계.
- `model.joblib`: Batch 1 전체로 재학습한 모델·입력 목록·예측 시점.
- `predictions.csv`, `error_analysis.csv`: 셀별 예측과 오류 분석.
- `auxiliary/`: 처리 가정·라벨 민감도 결과.

02 노트북 실행 후 이 노트북을 실행하면 모든 결과를 다시 생성합니다.

In [9]:
import joblib
artifact = joblib.load(ROOT / 'results/model.joblib')
np.testing.assert_allclose(artifact['model'].predict(error[artifact['features']]), error.prediction)
print('저장 모델의 Batch 2 예측 재현 확인 완료')

저장 모델의 Batch 2 예측 재현 확인 완료
